# Theorem 1 synthetic regenerative 3-gram experiment

This notebook studies theorem-1-style support drift in a synthetic token world.

We keep two sizes separate:
- `V`: vocabulary size
- `T`: text length

The run starts from a synthetic latent 1/2/3-gram system, generates an initial text of length `T`, and then iterates:
1. keep a retained block of length `(1 - alpha)T`
2. generate `alpha T` replacement tokens from the empirical trigram model of the current text
3. rebuild observed 1/2/3-grams from the new text
4. measure vocabulary size and the number of distinct 2-grams and 3-grams

The run is checkpointed and versioned on disk so it can be resumed if the notebook stops.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if p.name == "Drift_and_selection":
            return p
        if (p / "GitHub").exists() and (p / "Nat_Paper").exists():
            return p
    return start


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "GitHub" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from drift_selection.theorem1_synthetic_regeneration import (
    RegenerationConfig,
    SyntheticGrammarConfig,
    default_output_root,
    run_synthetic_regeneration,
)

pd.set_option("display.max_columns", 100)


## Configuration

This keeps the main structural parameters visible:
- `V`: token vocabulary size
- `T`: text length used to rebuild the empirical model each generation
- `alpha`: fraction replaced each generation
- `restart_probability`: optional restart rate inside the generated replacement segment


In [ ]:
EXPERIMENT_VERSION = "V0_01"
SEED = 123

V = 100
T = 1000
ALPHA = 0.25
GENERATIONS = 20
RESTART_PROBABILITY = 0.0

RUN_NAME = f"synthetic_regeneration_v{V}_t{T}_alpha_{str(ALPHA).replace('.', 'p')}"
OUTPUT_ROOT = default_output_root(PROJECT_ROOT)

grammar_cfg = SyntheticGrammarConfig(
    vocab_size=V,
    keep_prob_2gram=0.50,
    keep_prob_3gram=0.25,
    exact_support=False,
    support_cache_limit=4096,
)

regen_cfg = RegenerationConfig(
    text_length=T,
    generations=GENERATIONS,
    alpha=ALPHA,
    restart_probability=RESTART_PROBABILITY,
    sample_retained_block=True,
)

RESUME_RUN = True
FORCE_REBUILD = False


## Run or resume

This writes a versioned run directory with a manifest, checkpoint state, generation snapshots, partial/final metrics, and saved figures.

In [ ]:
run = run_synthetic_regeneration(
    version=EXPERIMENT_VERSION,
    run_name=RUN_NAME,
    grammar_config=grammar_cfg,
    regeneration_config=regen_cfg,
    seed=SEED,
    output_root=OUTPUT_ROOT,
    resume=RESUME_RUN,
    force_rebuild=FORCE_REBUILD,
    progress_bar=True,
)

results = pd.DataFrame(run.metrics_rows).sort_values("generation").reset_index(drop=True)

display(Markdown(f"**Run directory:** `{run.paths.run_dir}`"))
display(Markdown(f"**Checkpoint:** `{run.paths.checkpoint_path}`"))
display(Markdown(f"**Metrics CSV:** `{run.paths.metrics_final_path}`"))
display(Markdown(f"**Summary JSON:** `{run.paths.summary_path}`"))

results


In [ ]:
results[[
    "generation",
    "vocab_size",
    "distinct_2grams",
    "distinct_3grams",
    "vocab_ratio_vs_gen0",
    "distinct_2gram_ratio_vs_gen0",
    "distinct_3gram_ratio_vs_gen0",
    "token_entropy_bits",
]]


## Saved figures

The plots are saved to disk with version tags and timestamps in the titles so the experimental state survives the notebook session.

In [ ]:
display(Image(filename=str(run.paths.figures_dir / f"{EXPERIMENT_VERSION.lower()}_support_counts.png")))
display(Image(filename=str(run.paths.figures_dir / f"{EXPERIMENT_VERSION.lower()}_relative_support.png")))


In [ ]:
display(Markdown("**Saved sample texts**"))
print(run.paths.sample_text_path.read_text(encoding="utf-8"))


## Next extension toward theorem 2

The natural next step is to keep the same regenerative loop but replace plain replacement generation with a look-ahead publisher that prefers desirable `r`-grams and vetoes undesirable ones. That can reuse the same checkpointed run layout and the same distinction between `V` and `T`.